# HW2 — Technical Indicators & Strategy Backtesting

Interactive walkthrough of the backtesting platform built in `hw2/src`.

1. Pull 5+ years of daily OHLCV from Alpaca
2. Compute technical indicators
3. Run Trend-Following, Mean-Reversion, and a Custom strategy vs. Buy & Hold
4. Compare performance metrics and visualize equity curves, drawdowns, and signals

> Requires a `.env` in the repo with `ALPACA_API_KEY` / `ALPACA_SECRET_KEY`.
> Run `python run_backtest.py` from `hw2/scripts/` to regenerate every chart and the PDF report.

In [ ]:
import sys
sys.path.insert(0, '..')  # make hw2/src importable

from src.data_loader import load_daily_data
from src.indicators import add_all_indicators
from src import strategies as strat
from src.backtest import run_backtest
from src.metrics import metrics_table, format_metrics_table
from src import visualize as viz
from src.report import CHART_CONFIG

%matplotlib inline

## 1. Historical data
Select a ticker and pull daily bars. Try `AAPL`, `MSFT`, `SPY`, `QQQ`, `NVDA`.

In [ ]:
TICKER = 'SPY'
df = load_daily_data(TICKER, years=6)
print(f'{len(df)} bars  {df.index[0].date()} -> {df.index[-1].date()}')
df.tail()

## 2. Technical indicators
`add_all_indicators` attaches SMA/EMA/MACD/ADX (trend), RSI/Stochastic/Williams %R (momentum), Bollinger Bands/ATR (volatility), and OBV/CMF (volume).

In [ ]:
ind = add_all_indicators(df)
ind[['close','sma_50','macd','adx','rsi','bb_upper','bb_lower','cmf']].tail()

## 3 & 4. Strategies + backtesting engine
Each strategy returns a long-only 0/1 position series; the engine enters on the next bar (no look-ahead), $100k initial capital, no leverage/shorting.

In [ ]:
results = {}
for name, fn in strat.STRATEGIES.items():
    results[name] = run_backtest(df, fn(ind), name=name)
    print(f'{name:16s} trades={len(results[name].trades):3d} final=${results[name].final_value:,.0f}')

## 5. Performance metrics

In [ ]:
table = metrics_table(results)
format_metrics_table(table)

## 6. Visualizations
### Price + indicators + buy/sell signals

In [ ]:
for name in ['Trend Following', 'Mean Reversion', 'Custom']:
    cfg = CHART_CONFIG[name]
    viz.plot_price_signals(ind, results[name], title=f'{name} — {TICKER}',
                           overlays=cfg['overlays'], lower=cfg['lower']);

### Equity curve & drawdown comparison

In [ ]:
viz.plot_equity_curves(results, title=f'Equity Curve Comparison — {TICKER}');
viz.plot_drawdowns(results, title=f'Drawdown Comparison — {TICKER}');

## 7. Final report
Regenerate the full PDF (charts + tables + discussion) into `hw2/report/`:

In [ ]:
import os
from src.report import build_report
os.makedirs('../report', exist_ok=True)
path = build_report(f'../report/HW2_Report_{TICKER}.pdf', TICKER, ind, results)
print('wrote', path)